In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageOps

# Check for RTX 3050
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 1. Create Vocabulary (0 is reserved for CTC Blank)
ALPHABET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ "
char_to_int = {char: i + 1 for i, char in enumerate(ALPHABET)}
int_to_char = {i + 1: char for i, char in enumerate(ALPHABET)}
NUM_CLASSES = len(ALPHABET) + 1 

class WordDataset(Dataset):
    def __init__(self, txt_file, img_dir):
        self.img_dir = img_dir
        self.data = []
        
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    filename = parts[0] + ".png" 
                    text = " ".join(parts[1:]) 
                    self.data.append((filename, text))
                    
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename, text = self.data[idx]
        img_path = os.path.join(self.img_dir, filename)
        
        try:
            im = Image.open(img_path).convert('L')
        except FileNotFoundError:
            im = Image.new('L', (128, 32), color=255) 
            text = ""

        # Resize to Height=32, Width=variable
        cur_width, cur_height = im.size
        new_width = int(cur_width * (32 / cur_height))
        im = im.resize((new_width, 32), Image.LANCZOS)
        
        # Pad or Crop Width to 128
        if new_width < 128:
            padding = (0, 0, 128 - new_width, 0)
            im = ImageOps.expand(im, padding, fill=255)
        else:
            im = im.crop((0, 0, 128, 32))
            
        img_tensor = self.transform(im)
        
        # Convert text to integers
        encoded_text = [char_to_int[c] for c in text if c in char_to_int]
        target_tensor = torch.tensor(encoded_text, dtype=torch.long)
        
        return img_tensor, target_tensor

def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, 0)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.cat(targets) 
    return images, targets, target_lengths

# Initialize Loader
dataset = WordDataset('dataset/words_new.txt', 'dataset/words/')
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
print(f"Loaded {len(dataset)} words for training.")

In [ ]:
class HandwrittenWordReader(nn.Module):
    def __init__(self, num_chars, hidden_size=256, num_layers=2):
        super(HandwrittenWordReader, self).__init__()
        
        # Vision Brain (CNN)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d((2, 2), (2, 1), (0, 1)) 
        )
        
        # Sequence Brain (RNN)
        self.rnn = nn.LSTM(
            input_size=256 * 4, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            bidirectional=True, 
            batch_first=True
        )
        
        self.fc = nn.Linear(hidden_size * 2, num_chars)

    def forward(self, x):
        conv_out = self.cnn(x)
        batch, channels, height, width = conv_out.size()
        conv_out = conv_out.view(batch, channels * height, width).permute(0, 2, 1) 
        
        rnn_out, _ = self.rnn(conv_out)
        output = self.fc(rnn_out)
        
        output = F.log_softmax(output, dim=2).permute(1, 0, 2)
        return output

model = HandwrittenWordReader(num_chars=NUM_CLASSES).to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Alignment Brain (CTC)
criterion = nn.CTCLoss(blank=0, zero_infinity=True) 

epochs = 50 # Adjust based on how fast the loss drops

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, targets, target_lengths in train_loader:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        preds = model(images)
        seq_length = preds.size(0)
        batch_size = images.size(0)
        
        input_lengths = torch.full(size=(batch_size,), fill_value=seq_length, dtype=torch.long)
        
        loss = criterion(preds, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Training Loss: {running_loss/len(train_loader):.4f}")

print("Training Complete!")
torch.save(model.state_dict(), 'crnn_word_reader.pth')
print("Model saved to 'crnn_word_reader.pth'")